# 03b — SQL, transactions, and schema design — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/03.md) · [Course map](../PLAN.md) · [Project](../../projects/stage03/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Write joins, CTEs, and window queries
- Parameterize user values
- Use transactions and constraints


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

SQL describes desired results over tables. WHERE filters rows before grouping; HAVING filters groups. INNER JOIN requires a match; LEFT JOIN preserves unmatched left rows. UNION removes duplicates; UNION ALL preserves them. A CTE names an intermediate query; a subquery embeds a query inside another. Window functions compute across related rows without collapsing them.

Primary keys identify rows. Foreign keys enforce relationships. Normalization separates facts so updates do not produce contradictions. Indexes trade write/storage cost for faster access paths; inspect a query plan. A transaction groups operations atomically. Never construct SQL by interpolating user values; bind parameters instead. Schema migrations are versioned changes with a tested rollout strategy.

These exercises use SQLite for an offline first step. The project repeats them in PostgreSQL, where concurrency, types, query plans, and migration behavior need direct testing.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.executescript("CREATE TABLE orders(id INTEGER PRIMARY KEY, customer TEXT, amount INTEGER); INSERT INTO orders VALUES(1,'A',10),(2,'A',20),(3,'B',7);")
print(conn.execute("SELECT customer, SUM(amount) FROM orders GROUP BY customer").fetchall())


## Exercise 1: Bound filter

Return (id,amount) rows for a customer ordered by id, using SQL parameters.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def customer_orders(conn, customer):
    raise NotImplementedError


In [ ]:
assert customer_orders(conn, "A") == [(1, 10), (2, 20)]
assert customer_orders(conn, "' OR 1=1 --") == []
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

The SQL structure stays constant.

</details>


## Exercise 2: Running totals

Return (id,running_amount) per customer, ordered by id. Use SUM OVER with an explicit ROWS frame.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def running_totals(conn):
    raise NotImplementedError


In [ ]:
assert running_totals(conn) == [(1, 10), (2, 30), (3, 7)]
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Partition by customer, order by id.

</details>


## Exercise 3: Atomic batch

Insert rows inside a transaction; any duplicate key rolls back the entire batch.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def insert_batch(conn, rows):
    raise NotImplementedError


In [ ]:
expect_error(sqlite3.IntegrityError, lambda: insert_batch(conn, [(4, "C", 4), (1, "D", 5)]))
assert conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 3
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

The connection context manager rolls back on an exception.

</details>


## Independent transfer

Design customers, records, and categories tables with keys; write INNER/LEFT joins, CTE, HAVING, UNION, and correlated subquery examples. Use PostgreSQL EXPLAIN before/after an index and demonstrate rollback.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why a LEFT JOIN?

2. Are indexes free?


## Reading and review

- [Primary reference 1](https://www.postgresql.org/docs/current/tutorial.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
